In [ ]:
import pandas as pd
import numpy as np

import plotly.express as px
import plotly.graph_objs as go

import utility_functions as uf

In [ ]:
domain_id_list = uf.df_topics.domain_id.drop_duplicates().sort_values().to_list()
domain_list = []
for domain in domain_id_list:
    field_id_list = uf.df_topics.query(f"domain_id == {domain}").field_id.drop_duplicates().sort_values().to_list()
    field_list = []
    for field in field_id_list:
        subfield_id_list = uf.df_topics.query(f"field_id == {field}").subfield_id.drop_duplicates().sort_values().to_list()
        subfield_list = [{"id": subfield, "length": 1} for subfield in subfield_id_list]
        field_list.append({
            "id": field,
            "length": 10,
            "branches": subfield_list,
        })
    domain_list.append({
        "id": domain,
        "length": 100,
        "branches": field_list,
    })

subfields_tree = {
    "id": 0,
    "branches": domain_list,
}

# subfields_tree

# Read metrics_data

In [ ]:
path = "data/"
df_country_subfield_norm_world_norm = pd.read_csv(path+"df_country_subfield_norm_world.csv", index_col="country")

# W1 - distance

In [ ]:
def get_dist_w1_tree(tree, mu_dict, nu_dict):
    subtree = tree.get("branches", None)
    if subtree is None:
        leave_id = tree["id"]
        edge_length = tree.get("length", None)
        mu_id = mu_dict.get(str(leave_id), 0)
        nu_id = nu_dict.get(str(leave_id), 0)
        return mu_id, nu_id, abs(mu_id - nu_id) * edge_length
    else:
        mu_id_array = np.full(len(subtree), 0, dtype=float)
        nu_id_array = np.full(len(subtree), 0, dtype=float)
        dist_sum = 0
        edge_length = tree.get("length", None)
        for i, branch in enumerate(subtree):
            mu_id_array[i], nu_id_array[i], dist_branch = get_dist_w1_tree(branch, mu_dict, nu_dict)
            dist_sum += dist_branch
        if edge_length is None:
            return mu_id_array.sum(), nu_id_array.sum(), dist_sum
        else:
            return mu_id_array.sum(), nu_id_array.sum(), dist_sum + abs(mu_id_array.sum() - nu_id_array.sum()) * edge_length


In [ ]:
get_dist_w1_tree(subfields_tree,
                 mu_dict = df_country_subfield_norm_world_norm.loc["RU"].dropna().to_dict(),
                 nu_dict = df_country_subfield_norm_world_norm.loc["UA"].dropna().to_dict())

# Get country distances

In [ ]:
country_list = df_country_subfield_norm_world_norm.index.to_list()
top_20_countries_list = uf.top_n_countries_by_articles(20)
top_20_idx = [country_list.index(c) for c in top_20_countries_list]

In [ ]:
country_dist_w1 = np.full((len(country_list), len(country_list)), np.nan, dtype=float)

for i, country1 in enumerate(country_list):
    for j, country2 in enumerate(country_list):
        if i == j:
            country_dist_w1[i, j] = 0
        if i < j:
            _, _, country_dist_w1[i, j] = _, _, country_dist_w1[j, i] = get_dist_w1_tree(
                subfields_tree,
                mu_dict = df_country_subfield_norm_world_norm.loc[country1].dropna().to_dict(),
                nu_dict = df_country_subfield_norm_world_norm.loc[country2].dropna().to_dict())

In [ ]:
df_country_dist_w1 = pd.DataFrame(country_dist_w1, index=country_list, columns=country_list)
# df_country_dist_w1.to_csv(path+"df_country_dist_w1.csv")

In [ ]:
df_country_dist_w1

In [ ]:
df_country_dist_w1_norm = (
    df_country_dist_w1
    .div(df_country_dist_w1.sum(axis=1), axis=0)
)

In [ ]:
# x_labels = list(map(str, df_country_dist_w1.columns))
# y_labels = list(map(str, df_country_dist_w1.index))

sub_list = list(map(str, top_20_countries_list))

# country = "AX"
# top_n = 20
# sub_list = df_country_dist_w1.sort_values(country, ascending=True)[[country]].head(top_n).index.to_list()

x_labels = sub_list
y_labels = sub_list

fig = go.Figure(
    data=go.Heatmap(
        # z=df_country_dist_w1.values,
        z=df_country_dist_w1.loc[sub_list, sub_list].values,
        x=x_labels,
        y=y_labels,
        colorscale="Viridis",
        zmin=0,
        zmax=200,
    )
)

# Layout
fig.update_layout(
    title="Distances",
    width=1000,
    # height=20*len(y_labels),  # taller if more rows
    height=40*len(y_labels),  # taller if more rows
    margin=dict(l=100, r=50, t=50, b=100),
    xaxis=dict(tickangle=45, automargin=True),
    yaxis=dict(autorange="reversed", automargin=True)  # keep top-to-bottom ordering
)

fig.show()

In [ ]:
uf.get_country_info("SS")

In [ ]:
country = "AX"
top_n = 20
df_plot = df_country_dist_w1.sort_values(country, ascending=True)[[country]].head(top_n)
px.bar(df_plot, x=country, title=uf.get_country_info(country).iloc[0, 0])

In [ ]:
top_n = 10
countries = uf.top_n_countries_by_articles(top_n)

# Create figure
fig = go.Figure()

for country in countries:
    df_plot = df_country_dist_w1.sort_values(country, ascending=True).head(top_n)
    fig.add_trace(
        go.Bar(
            x=df_plot[country],           # value
            y=np.arange(top_n),
            hovertext=[code+" "+uf.id2name_country[code] for code in df_plot.index],# category
            name=uf.get_country_info(country).iloc[0, 0],
            orientation='h'
        )
    )

# Layout
fig.update_layout(
    barmode='group',       # 'stack' if you prefer stacked bars
    title=f"Top {top_n} categories for selected countries",
    xaxis_title="Value",
    yaxis_title=None,
    height=800,
    margin=dict(l=150),    # leave space for long y labels
)

# Optional: largest bar at top
fig.update_yaxes(autorange="reversed")

fig.show()

In [24]:
df_country_dist_w1_stats = (
    df_country_dist_w1
    .mean()
    .to_frame("mean")
    .assign(median=df_country_dist_w1.median(),
            std=df_country_dist_w1.std(),
            range=df_country_dist_w1.max() - df_country_dist_w1.min(),
            gini=df_country_dist_w1.apply(uf.gini, axis=1),)
)
df_country_dist_w1_stats

,mean,median,std,range,gini
AD,96.359117,88.033964,37.305348,222.000000,0.205167
AE,67.336031,57.272726,38.895805,187.905810,0.315114
AF,90.448473,83.759610,33.646812,196.960827,0.202394
AG,126.957092,128.810299,36.587335,222.000000,0.157735
AL,67.539230,58.967517,36.926343,185.047922,0.300277
...,...,...,...,...,...
XK,80.308110,72.019106,39.610311,194.470032,0.273385
YE,69.458481,57.955498,38.527637,188.823446,0.297314
ZA,75.588093,69.329120,38.699958,182.939647,0.280611
ZM,121.519193,123.374093,36.287644,214.554405,0.165022


In [25]:
px.histogram(df_country_dist_w1_stats, x="range", height=800)